# **FinIntel AI: RAG-Based Financial Decision System**

**Project Introduction**

FinIntel AI is an intelligent fraud detection system that combines Machine Learning, Retrieval-Augmented Generation (RAG), and Large Language Models (LLMs) to detect fraudulent financial transactions and generate explainable AI-driven fraud insights. The system uses semantic retrieval and NVIDIA NIM APIs to provide contextual fraud reasoning and actionable recommendations.

**Problem Statement**

Traditional fraud detection systems only classify transactions as fraud or non-fraud without explaining the reason behind the prediction. This lack of explainability makes fraud investigation slow and difficult for analysts. The project aims to build an AI-powered fraud analysis system capable of both fraud detection and intelligent reasoning.

**Project Objectives**

- Detect fraudulent financial transactions using Machine Learning models.
- Handle imbalanced fraud datasets using SMOTE and anomaly detection techniques.
- Build a RAG pipeline using embeddings and ChromaDB for semantic fraud retrieval.
- Integrate NVIDIA NIM LLM APIs for AI-generated fraud explanations.
- Provide explainable, context-aware, and actionable fraud investigation insights.

**Project Pipeline**

Data Collection →
EDA & Feature Engineering →
SMOTE & Anomaly Handling →
Random Forest Fraud Detection →
Fraud Embeddings Generation →
ChromaDB Vector Storage →
Semantic Similarity Retrieval →
NVIDIA NIM LLM Integration →
AI Fraud Reasoning & Recommendations →
Streamlit Deployment

# **Phase 1 - RAG Pipeline Fundamentals (Dummy examples)**

In [ ]:
# STEP 1 — Install libraries

!pip install --upgrade sentence-transformers datasets chromadb

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

In [ ]:
# STEP 2 — Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# STEP 3 — Create sample data
documents = [
    "Fraud transactions often involve unusually high amounts",
    "Transactions at midnight are considered risky",
    "Frequent small transactions may indicate fraud",
    "Normal transactions follow regular patterns"
]

In [ ]:
# STEP 4 — Convert text to embeddings
embeddings = model.encode(documents)

In [ ]:
# STEP 5 — Initialize Chroma DB
client = chromadb.Client()
collection = client.create_collection(name = "fraud_data", get_or_create=True)

In [ ]:
# STEP 6 — Store embeddings
collection.add(
    documents= documents,
    embeddings = embeddings.tolist(),
    ids = ['1', '2', '3', '4']
)

In [ ]:
# STEP 7 — Query (Retrieval)
query = "high value transaction fraud"
query_embedding = model.encode([query])

In [ ]:
# STEP 8 — Search similar content
results = collection.query(
    query_embeddings = query_embedding.tolist(),
    n_results = 2
)

print(results['documents'])

[['Fraud transactions often involve unusually high amounts', 'Frequent small transactions may indicate fraud']]


# **Phase 2 - LLM Integration & AI Generation Layer**

**Goal of this phase is:**

To take retrieved data → pass to LLM → generate:

- explanation
- risk insight
- recommendation

User Query → Retrieve Context → LLM → Final Answer

In [ ]:
# STEP 1 — Install Library
!pip install transformers

# Used transformers as it provides pretrained LLMs, text generation pipelines, tokenizer + model loading

In [ ]:
# STEP 2 — Import Pipeline
from transformers import pipeline

# pipeline() simplifies loading model, inference, generation

In [ ]:
# STEP 3 — Load Generation Model
generator = pipeline(
    "text-generation",
     model="gpt2"
)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# STEP 4 — Prompt Engineering

prompt = """
You are a financial fraud analyst.

Question:
Why is this transaction risky?

Answer clearly.
"""

In [ ]:
# STEP 5 — Generate response

response = generator(
    prompt,
    max_new_tokens=80,
)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
print(response[0]['generated_text'])


You are a financial fraud analyst.

Question:
Why is this transaction risky?

Answer clearly.

It is not a business transaction.

If you are a financial fraud analyst you have no business being involved.

This is your chance to help out the financial fraud community.

It is not a business transaction.

If you are a financial fraud analyst you probably want to avoid this transaction.

But, what if you are not?

You have been warned


**Conclusion** - During this phase i.e LLM integration, lightweight local generation models are tested for explanation generation. However, due to limitations in instruction-following and response quality, the architecture is planned to be upgraded using hosted inference APIs like NVIDIA NIM for better scalability and output quality.

# **Phase 3 - Real Financial RAG Pipeline**

**Goal of this phase:**

Fraud Dataset →
Convert Rows to Text →
Create Embeddings →
Store in ChromaDB →
Retrieve Similar Fraud Cases

In [5]:
# STEP 1 — Install Libraries

!pip install sentence_transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.4 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentel

In [6]:
import pandas as pd
import joblib
from sentence_transformers import SentenceTransformer
import chromadb

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
# STEP 3 — Load Dataset

df = pd.read_csv("/content/drive/MyDrive/FinIntel resources/creditcard.csv")

In [9]:
rf_model = joblib.load("/content/drive/MyDrive/FinIntel resources/rf_model (1).pkl")

In [24]:
# Data Cleaning

df.shape

(283726, 31)

In [25]:
df = df.drop_duplicates()

In [26]:
df.shape

(283726, 31)

In [27]:
# STEP 4 — Extract Fraud Transactions

fraud_df = df[df['Class'] == 1]

# Insights- We only want fraud-related knowledge because RAG should retrieve meaningful fraud insights

In [28]:
fraud_df

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
541,406.0,-2.312227,1.951992,-1.609851,3.997906,-0.522188,-1.426545,-2.537387,1.391657,-2.770089,...,0.517232,-0.035049,-0.465211,0.320198,0.044519,0.177840,0.261145,-0.143276,0.00,1
623,472.0,-3.043541,-3.157307,1.088463,2.288644,1.359805,-1.064823,0.325574,-0.067794,-0.270953,...,0.661696,0.435477,1.375966,-0.293803,0.279798,-0.145362,-0.252773,0.035764,529.00,1
4920,4462.0,-2.303350,1.759247,-0.359745,2.330243,-0.821628,-0.075788,0.562320,-0.399147,-0.238253,...,-0.294166,-0.932391,0.172726,-0.087330,-0.156114,-0.542628,0.039566,-0.153029,239.93,1
6108,6986.0,-4.397974,1.358367,-2.592844,2.679787,-1.128131,-1.706536,-3.496197,-0.248778,-0.247768,...,0.573574,0.176968,-0.436207,-0.053502,0.252405,-0.657488,-0.827136,0.849573,59.00,1
6329,7519.0,1.234235,3.019740,-4.304597,4.732795,3.624201,-1.357746,1.713445,-0.496358,-1.282858,...,-0.379068,-0.704181,-0.656805,-1.632653,1.488901,0.566797,-0.010016,0.146793,1.00,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279863,169142.0,-1.927883,1.125653,-4.518331,1.749293,-1.566487,-2.010494,-0.882850,0.697211,-2.064945,...,0.778584,-0.319189,0.639419,-0.294885,0.537503,0.788395,0.292680,0.147968,390.00,1
280143,169347.0,1.378559,1.289381,-5.004247,1.411850,0.442581,-1.326536,-1.413170,0.248525,-1.127396,...,0.370612,0.028234,-0.145640,-0.081049,0.521875,0.739467,0.389152,0.186637,0.76,1
280149,169351.0,-0.676143,1.126366,-2.213700,0.468308,-1.120541,-0.003346,-2.234739,1.210158,-0.652250,...,0.751826,0.834108,0.190944,0.032070,-0.739695,0.471111,0.385107,0.194361,77.89,1
281144,169966.0,-3.113832,0.585864,-5.399730,1.817092,-0.840618,-2.943548,-2.208002,1.058733,-1.632333,...,0.583276,-0.269209,-0.456108,-0.183659,-0.328168,0.606116,0.884876,-0.253700,245.00,1


In [29]:
fraud_df.shape

(473, 31)

In [30]:
# STEP 5 — Select Important Features

fraud_df = fraud_df[['Time', 'Amount', 'V14', 'V10', 'Class']]

# Insights- Feature importance analysis from Random Forest showed that V10 and V14 was a stronger fraud indicator,
#           so it was incorporated into the RAG knowledge representation for better contextual retrieval.

In [31]:
# STEP 6 — Convert Rows into Natural Language

documents = []

for _,row in fraud_df.iterrows():

  text = f"""
Fraud Transaction Summary

• Transaction Time: {row['Time']}
• Transaction Amount: {row['Amount']}
• Suspicious V14 Score: {row['V14']}
• Suspicious V10 Score: {row['V10']}

This transaction shows abnormal fraud-related behavior.
  """

  documents.append(text)

# Insights - Converting tabular data → text because embedding models understand text better and this is called semantic transformation.

In [32]:
# STEP 7 — Check Sample Document

print(documents[0])


Fraud Transaction Summary

• Transaction Time: 406.0
• Transaction Amount: 0.0
• Suspicious V14 Score: -4.289253782
• Suspicious V10 Score: -2.772272145

This transaction shows abnormal fraud-related behavior.
  


In [33]:
fraud_df.head(1)

# Insights- Also, verifying it through the table

,Time,Amount,V14,V10,Class
541,406.0,0.0,-4.289254,-2.772272,1


In [34]:
# STEP 8 — Load Embedding Model

model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [35]:
# STEP 9 — Create Embeddings

embeddings = model.encode(documents)

In [36]:
client = chromadb.Client()

collection = client.create_collection(
    name = "fraud_rag",
    get_or_create=True
)

In [37]:
# STEP 11 — Store Documents + Embeddings

collection.add(
    documents = documents,
    embeddings = embeddings.tolist(),
    ids = [str(i) for i in range(len(documents))]
)

In [38]:
# STEP 12 — User Query

query = "high amount fraud transaction at unusual time"

In [39]:
# STEP 13 — Convert Query into Embedding

query_embedding = model.encode([query])

In [40]:
# STEP 14 — Retrieve Similar Fraud Cases

results = collection.query(
    query_embeddings = query_embedding.tolist(),
    n_results = 3
)

In [41]:
# STEP 15 — Show Results

for i, doc in enumerate(results['documents'][0], 1):

    print("=" * 50)
    print(f"Retrieved Fraud Case {i}")
    print("=" * 50)

    print(doc)

    print("\n")

Retrieved Fraud Case 1

Fraud Transaction Summary

• Transaction Time: 156710.0
• Transaction Amount: 7.59
• Suspicious V14 Score: 0.141331684
• Suspicious V10 Score: 1.081513989

This transaction shows abnormal fraud-related behavior.
  


Retrieved Fraud Case 2

Fraud Transaction Summary

• Transaction Time: 18675.0
• Transaction Amount: 188.78
• Suspicious V14 Score: -9.809881502
• Suspicious V10 Score: -12.93892931

This transaction shows abnormal fraud-related behavior.
  


Retrieved Fraud Case 3

Fraud Transaction Summary

• Transaction Time: 13323.0
• Transaction Amount: 1.0
• Suspicious V14 Score: -18.04999769
• Suspicious V10 Score: -13.13669837

This transaction shows abnormal fraud-related behavior.
  




In [42]:
# STEP 16 — Show Results with similarity search

results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=3
)

for i, (doc, distance) in enumerate(
    zip(results['documents'][0], results['distances'][0]), 1):

    print("=" * 50)
    print(f"Retrieved Fraud Case {i}")
    print("=" * 50)

    print(doc)

    print(f"\nSimilarity Distance Score: {distance}")

    print("\n")


# Insights- Lower distance = MORE similar which means
#           Fraud Case 1 = MOST relevant fraud pattern

Retrieved Fraud Case 1

Fraud Transaction Summary

• Transaction Time: 156710.0
• Transaction Amount: 7.59
• Suspicious V14 Score: 0.141331684
• Suspicious V10 Score: 1.081513989

This transaction shows abnormal fraud-related behavior.
  

Similarity Distance Score: 0.7211734056472778


Retrieved Fraud Case 2

Fraud Transaction Summary

• Transaction Time: 18675.0
• Transaction Amount: 188.78
• Suspicious V14 Score: -9.809881502
• Suspicious V10 Score: -12.93892931

This transaction shows abnormal fraud-related behavior.
  

Similarity Distance Score: 0.7214699983596802


Retrieved Fraud Case 3

Fraud Transaction Summary

• Transaction Time: 13323.0
• Transaction Amount: 1.0
• Suspicious V14 Score: -18.04999769
• Suspicious V10 Score: -13.13669837

This transaction shows abnormal fraud-related behavior.
  

Similarity Distance Score: 0.7222285270690918




**Conclusion**- Fraud cases are retrieved based on semantic similarity between query embeddings and stored fraud embeddings using vector similarity search.

# **Phase 4 - AI Reasoning & Decision Intelligence Layer**

**NVIDIA NIM API Integration**

**Testing the model**

In [43]:
import os

In [44]:
from google.colab import userdata

api_key = userdata.get('NVIDIA_API_KEY')

In [45]:
model="meta/llama-3.1-70b-instruct"

In [46]:
from openai import OpenAI

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=api_key
)

In [47]:
response = client.chat.completions.create(

    model="meta/llama-3.1-70b-instruct",

    messages=[
        {
            "role": "user",
            "content": "Explain why high-value financial transactions can be risky."
        }
    ],

    temperature=0.3,
    max_tokens=150
)

In [48]:
print(response.choices[0].message.content)

High-value financial transactions can be risky for several reasons:

1. **Increased exposure to fraud**: Large transactions often involve significant sums of money, making them attractive targets for scammers, hackers, and other malicious actors. The potential reward for successfully executing a fraudulent transaction is higher, which can lead to more sophisticated and aggressive attempts to compromise the transaction.
2. **Higher stakes for errors**: With high-value transactions, even small mistakes or errors can have significant consequences. A single mistake in the transaction details, such as an incorrect account number or routing code, can result in the loss of a substantial amount of money.
3. **Greater regulatory scrutiny**: High-value transactions are often subject to increased regulatory scrutiny, particularly in the context of anti-money laundering (AML


**Checking on real Dataset**

In [49]:
# STEP 1 — Import Libraries

from openai import OpenAI
from google.colab import userdata

In [50]:
# STEP 2 — Load NVIDIA API Key Securely

api_key = userdata.get('NVIDIA_API_KEY')

In [51]:
# STEP 3 — Connect NVIDIA NIM

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=api_key
)

In [53]:
# STEP 4 — Select New Transaction (first row of dataset)

sample_transaction = df.drop('Class', axis = 1).iloc[[0]]

In [55]:
# Creating hour column so that it matches the feature of the previous dataset

import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/FinIntel resources/creditcard.csv")

# Recreate engineered feature
df['Hour'] = df['Time'] // 3600

In [56]:
df.Hour

,Hour
0,0.0
1,0.0
2,0.0
3,0.0
4,0.0
...,...
284802,47.0
284803,47.0
284804,47.0
284805,47.0


In [57]:
prediction = rf_model.predict(df.drop('Class', axis = 1).iloc[[0]])

In [58]:
# STEP 6 — Show Prediction Result

if prediction[0] == 1:
   prediction_label = "Fraudulent Transaction"

else:
    prediction_label = "Normal Transaction"

print(prediction_label)

# Insights- As we picked the first row of dataset and dataset is highly imbalanced, fraud transactions are very rare.
#           So, normal transaction is actually expected

Normal Transaction


In [59]:
# Rechecking

print(df.iloc[0]['Class'])

# Insights- Class belongs to 0 which means this transaction is normal.

0.0


In [60]:
# STEP 7 — Convert Transaction into Text

transaction_text = f"""
Transaction Summary

• Transaction Amount: {sample_transaction['Amount'].values[0]}
• Transaction Time: {sample_transaction['Time'].values[0]}
• Suspicious V14 Score: {sample_transaction['V14'].values[0]}
• Suspicious V10 Score: {sample_transaction['V10'].values[0]}
"""

# Insights- Converted structured transaction → semantic text

In [61]:
from sentence_transformers import SentenceTransformer

# STEP 8 — Create Query Embedding

model = SentenceTransformer('all-MiniLM-L6-v2') # Re-initialize the SentenceTransformer model
query_embedding = model.encode([transaction_text])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [62]:
# STEP 9 — Retrieve Similar Cases

results = collection.query(
    query_embeddings = query_embedding.tolist(),
    n_results = 3
)

In [63]:
# STEP 10 — Combine Retrieved Cases

retrieved_cases = "\n\n".join(results['documents'][0])

In [64]:
# STEP 11 — Create FINAL AI Prompt

prompt = f"""
You are an expert financial fraud analyst.

A new transaction has been analyzed.

Prediction Result:
{prediction_label}

Transaction Details:
{transaction_text}

Retrieved Similar Fraud Cases:
{retrieved_cases}

Based on the transaction and retrieved fraud patterns, generate:

1. Fraud Reason
2. Risk Level
3. Suggested Action
4. Investigation Summary

Keep the response professional and concise.
"""

In [65]:
# STEP 12 — Call NVIDIA NIM

response = client.chat.completions.create(
    model = "meta/llama-3.1-70b-instruct",
    messages = [
        {
            "role" : "user",
            "content" : prompt
        }
    ],

    temperature = 0.3,
    max_tokens = 300
)

In [67]:
# STEP 13 — Print Final AI Explanation

print(response.choices[0].message.content)

# Insights- Trnsaction is normal and risk factor is low. As suggested in the output, no immediate action is required.

**Transaction Analysis Report**

**1. Fraud Reason:**
The transaction in question does not exhibit typical characteristics of a fraudulent transaction. However, based on the retrieved similar fraud cases, it appears that the suspicious V14 and V10 scores are not significantly high to indicate a potential fraud. Nevertheless, the presence of similar fraud cases with low transaction amounts and high suspicious scores suggests that the transaction may be part of a larger scheme to test the system or evade detection.

**2. Risk Level:**
Low to Moderate

**3. Suggested Action:**
Monitor the account activity closely for any subsequent transactions that may indicate a larger pattern of fraudulent behavior. Verify the transaction details with the customer to ensure its legitimacy. No immediate action is required, but continued surveillance is necessary to assess the risk.

**4. Investigation Summary:**
The transaction amount of $149.62 and transaction time of 0.0 do not raise significant conce

**Extracting Fraud cases from Dataset**

In [68]:
fraud_rows = df[df['Class'] == 1]

print(fraud_rows.head())

        Time        V1        V2        V3        V4        V5        V6  \
541    406.0 -2.312227  1.951992 -1.609851  3.997906 -0.522188 -1.426545   
623    472.0 -3.043541 -3.157307  1.088463  2.288644  1.359805 -1.064823   
4920  4462.0 -2.303350  1.759247 -0.359745  2.330243 -0.821628 -0.075788   
6108  6986.0 -4.397974  1.358367 -2.592844  2.679787 -1.128131 -1.706536   
6329  7519.0  1.234235  3.019740 -4.304597  4.732795  3.624201 -1.357746   

            V7        V8        V9  ...       V22       V23       V24  \
541  -2.537387  1.391657 -2.770089  ... -0.035049 -0.465211  0.320198   
623   0.325574 -0.067794 -0.270953  ...  0.435477  1.375966 -0.293803   
4920  0.562320 -0.399147 -0.238253  ... -0.932391  0.172726 -0.087330   
6108 -3.496197 -0.248778 -0.247768  ...  0.176968 -0.436207 -0.053502   
6329  1.713445 -0.496358 -1.282858  ... -0.704181 -0.656805 -1.632653   

           V25       V26       V27       V28  Amount  Class  Hour  
541   0.044519  0.177840  0.261145 -

In [70]:
print(df.iloc[541]['Class'])

# Insights- The transaction is fraud.

1.0


**Now, Evaluating for Fraudlent data**

In [71]:
fraud_rows = df[df['Class'] == 1]

sample_transaction = fraud_rows.drop(
    'Class',
    axis=1
).iloc[[0]]

In [72]:
sample_transaction = fraud_rows.drop(
    'Class',
    axis=1
).iloc[[0]]

In [75]:
# Prediction

prediction = rf_model.predict(sample_transaction)

print(prediction)

[1]


In [76]:
print(fraud_rows.index[0])

# Insights- Now, we have first row as a fraud data.

541


In [77]:
# Show Prediction Result

if prediction[0] == 1:
   prediction_label = "Fraudulent Transaction"

else:
    prediction_label = "Normal Transaction"

print(prediction_label)

Fraudulent Transaction


In [78]:
# Transaction Text

transaction_text = f"""
Transaction Summary

• Transaction Amount: {sample_transaction['Amount'].values[0]}
• Transaction Time: {sample_transaction['Time'].values[0]}
• Suspicious V14 Score: {sample_transaction['V14'].values[0]}
• Suspicious V10 Score: {sample_transaction['V10'].values[0]}
"""

In [80]:
# from sentence_transformers import SentenceTransformer

# Embedding

# model = SentenceTransformer('all-MiniLM-L6-v2')
query_embedding = model.encode([transaction_text])

In [81]:
# Retrieval

results = collection.query(
    query_embeddings = query_embedding.tolist(),
    n_results = 3
)

In [82]:
# Retrieved Cases

retrieved_cases = "\n\n".join(results['documents'][0])

In [83]:
# Prompt

prompt = f"""
You are an expert financial fraud analyst.

A new transaction has been analyzed.

Prediction Result:
{prediction_label}

Transaction Details:
{transaction_text}

Retrieved Similar Fraud Cases:
{retrieved_cases}

Based on the transaction and retrieved fraud patterns, generate:

1. Fraud Reason
2. Risk Level
3. Suggested Action
4. Investigation Summary

Keep the response professional and concise.
"""

In [84]:
# NIM Call

response = client.chat.completions.create(
    model = "meta/llama-3.1-70b-instruct",
    messages = [
        {
            "role" : "user",
            "content" : prompt
        }
    ],

    temperature = 0.3,
    max_tokens = 300
)

In [86]:
# Final Output

print(response.choices[0].message.content)

# Insights- Immediate action need to be taken as the transaction is fraud showing abnormal behaviour. The risk level is high.

**Fraud Analysis Report**

**1. Fraud Reason:**
The transaction is suspected to be fraudulent due to its similarity to known fraud patterns, specifically exhibiting abnormal behavior in the Suspicious V14 and V10 scores. The transaction amount of 0.0 and identical transaction time (406.0) to one of the retrieved similar fraud cases further supports this assessment.

**2. Risk Level:**
High. The transaction's similarity to multiple known fraud cases and its abnormal behavior in the suspicious scores indicate a high likelihood of fraudulent activity.

**3. Suggested Action:**
Immediate action is recommended to prevent potential financial loss. The suggested course of action is to:
- Freeze the account associated with the transaction.
- Conduct a thorough investigation into the transaction and the account holder's activity.
- Verify the account holder's identity and transaction history.
- Consider notifying law enforcement if the investigation confirms fraudulent activity.

**4. Investiga

**Final Results**

- Achieved high fraud detection performance using Random Forest on imbalanced transaction data.
- Successfully implemented semantic fraud retrieval using embeddings and ChromaDB.
- Built an AI-powered reasoning layer using NVIDIA NIM APIs for contextual fraud explanations.
- Developed an end-to-end explainable fraud intelligence system with real-time AI-generated recommendations.
- Deployed the solution using Streamlit for interactive user access.

**Conclusion**

FinIntel AI demonstrates how Machine Learning, RAG architecture, Vector Databases, and LLMs can be combined to build an intelligent and explainable fraud detection system. The project enhances fraud investigation by providing contextual reasoning, semantic retrieval, and AI-generated recommendations, making the system more reliable and business-oriented than traditional fraud detection models.